# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**: Evan Sims and David Rha

**ID**: ems452 and dgr79

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [1]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `~/Documents/School/Senior/BEE 4750/hw5-rha-sims_hw5`
   Installed GR_jll ───────────── v0.73.18+0
   Installed OpenBLAS32_jll ───── v0.3.29+0
   Installed HiGHS_jll ────────── v1.12.0+0
   Installed PlotUtils ────────── v1.4.4
   Installed Measures ─────────── v0.3.3
   Installed MutableArithmetics ─ v1.6.7
   Installed OpenSSL ──────────── v1.6.0
   Installed FFMPEG ───────────── v0.4.5
   Installed Pango_jll ────────── v1.57.0+0
   Installed StaticArraysCore ─── v1.4.4
   Installed JSON ─────────────── v1.3.0
   Installed DataStructures ───── v0.19.3
   Installed Adapt ────────────── v4.4.0
   Installed GraphRecipes ─────── v0.5.15
   Installed StatsBase ────────── v0.34.8
   Installed StableRNGs ───────── v1.0.4
   Installed HiGHS ────────────── v1.20.1
   Installed METIS_jll ────────── v5.1.3+0
   Installed ForwardDiff ──────── v1.3.0
   Installed StructUtils ──────── v2.6.0
   Installed FFMPEG_jll ───────── v8.0.0+0
   Installed JuMP ─────────────── v1.29.

In [2]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.

-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

 Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1

Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

>Overall recycle fraction: 37.75%

>Overall ash fraction: 16.41%

>These were calculated by taking the dot product of the % of total mass vector and the combustion ash % or MRF recycle rate % vectors to get the weighted average of the overall percentage for each.

#### Problem 1.2

What are the decision variables for your optimization problem? Provide
notation and variable meaning.

The decision variables for are follows:

$W_{i,j}$ --> The amount of waste to send from each city ($i$) to each facility ($j$)

$R_{k,j}$ --> The amount of residual ash to send from each facility ($k$) to another facility ($j$) for disposal

$Y_j$ --> Binary variable for operational status (on/off)

For indexing, cities are labeled 1-3, and facilities are LF = 4, MRF = 5, and WTE = 6
for i = [1,2,3], k = [5,6], j = [4,5,6]

In [9]:
waste = Model(HiGHS.Optimizer)
I = [1:3]
K = [5:6]
J = [4:6]
@variable(waste, W[i in I, j in J], Int)
@variable(waste, R[k in K, j in J], Int)
@variable(waste, Y[j in J], Bin)

1-dimensional DenseAxisArray{VariableRef,1,...} with index sets:
    Dimension 1, UnitRange{Int64}[4:6]
And data, a 1-element Vector{VariableRef}:
 Y[4:6]

#### Problem 1.3

Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).

$FC_j$ is fixed costs per facility $j$ multiplied by the binary operational variable $Y_j$:
>$FC_j * Y_j$ = fixed cost portion of objective

$DC_j$ is dumping (tipping) costs per facility $j$ multiplied by the amount of waste going to facility $j$ which is sum of $i$ across $W_{i,j}$ and $k$ across $R_{k,j}$ with k not equal to j:
>$DC_j * (\sum_i W_{i,j} + \sum_k  R_{k,j})$ = variable dumping cost portion of objective

$RC_j$ is the recycle cost for facility $j$ (only $j=2$ for MRF will be nonzero) multiplied by $r_j$ the recycle rate for facility $j$ and multiplied again by the amount of waste going to facility $j$ which is sum of $i$ across $W_{i,j}$:
>$RC_j * r_j * \sum_i W_{i,j}$ = variable recycle cost portion of objective

$TC_i,j$ is transportation costs per Mg-km from facility/city $i$ to facility $j$ multiplied by the amount of waste going from facility/city $i$ to $j$ which is sum of $i$ across $W_{i,j}$ and multiplied by the distance $d_{i,j}$ from facility/city $i$ to facility $j$. Do the same for $R_{k,j}$ across k add that as well (with k not equal to j):
>$TC_j *  (\sum_i (d_{i,j} * W_{i,j}) + \sum_k (d_{k,j} * R_{k,j}))$ = variable transportation cost portion of objective


OVERALL OBJECTIVE FUNCTION:

$ min_{Y,W,R} \sum_j [(FC_j * Y_j) + (DC_j * (\sum_i W_{i,j} + \sum_k  R_{k,j})) + (RC_j * r_j * \sum_i W_{i,j}) + (TC_j *  (\sum_i (d_{i,j} * W_{i,j}) + \sum_k (d_{k,j} * R_{k,j})))] $

$min_{Y,W,R} 
[(FC_4 * Y_4) + (FC_5 * Y_5) + (FC_6 * Y_6) + 
                (DC_4 * (\sum_i W_{i,4} + R_{5,4} + R_{6,4})) + 
                (DC_5 * (\sum_i W_{i,5} + R_{6,5})) + 
                (DC_6 * (\sum_i W_{i,6} + R_{5,6})) +
                (RC_5 * r_5 * \sum_i W_{i,5}) + 
                (TC_4 *  (\sum_i (d_{i,4} * W_{i,4}) + (d_{5,4} * R_{5,4}) + (d_{6,4} * R_{6,4}))) + 
                (TC_5 *  (\sum_i (d_{i,5} * W_{i,5}) + (d_{6,5} * R_{6,5}))) +
                (TC_6 *  (\sum_i (d_{i,6} * W_{i,6}) + (d_{5,6} * R_{5,6})))]$

In [16]:
waste = Model(HiGHS.Optimizer)
I = 1:3
K = 5:6
J = 4:6
@variable(waste, W[i in I, j in J], Int)
@variable(waste, R[k in K, j in J], Int)
@variable(waste, Y[j in J], Bin)

FC = [2000, 1500, 2500] # $/day for correspond to j=4,5,6
DC = [50, 7, 60] # $/Mg for j=4,5,6
RC = 40 # $/Mg for j=5 because its the only recycle facility
TC = 1.5 # $/Mg/km for all travel between each city and facility or between two facilities
r = 0.3775 #recycle rate
a = 0.1641 #ash production rate

#distances from facility/city i (rows) to j (columns)
d = [
    0 0 0 5 30 15
    0 0 0 15 25 10
    0 0 0 13 45 20
    0 0 0 0 32 18
    0 0 0 32 0 15
    0 0 0 18 15 0
]

@objective(waste, Min, 
                    #fixed costs
                    FC[1]*Y[4] + FC[2]*Y[5] + FC[3]*Y[6] 
                    #dumping (tipping costs)
                    + DC[1]*(sum(W[i,4] for i in I)+R[5,4]+R[6,4])
                    + DC[2]*(sum(W[i,5] for i in I)+R[6,5])
                    + DC[3]*(sum(W[i,6] for i in I)+R[5,6])
                    #recycle costs
                    + RC*r*sum(W[i,5] for i in I)
                    #transportation costs
                    + TC*(sum(d[i,4]*W[i,4] for i in I) + d[5,4]*R[5,4] + d[6,4]*R[6,4])
                    + TC*(sum(d[i,5]*W[i,5] for i in I) + d[6,5]*R[6,5])
                    + TC*(sum(d[i,6]*W[i,6] for i in I) + d[5,6]*R[5,6])
)


2000 Y[4] + 1500 Y[5] + 2500 Y[6] + 57.5 W[1,4] + 72.5 W[2,4] + 69.5 W[3,4] + 98 R[5,4] + 77 R[6,4] + 67.1 W[1,5] + 59.6 W[2,5] + 89.6 W[3,5] + 29.5 R[6,5] + 82.5 W[1,6] + 75 W[2,6] + 90 W[3,6] + 82.5 R[5,6]

#### Problem 1.4

Derive all relevant constraints. Make sure to include any needed
justifications or derivations.



#### Problem 1.5

Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

#### Problem 1.6

Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data,
including ramping constraints for each generator, is provided in
\`data/generators.csv.’ In period 1, the demand is
$d_1 = 1100 \text{MW}$. In period 2, the demand is projected to be
$d_2 = 1200 \text{MW}$, but there is a 25% probability that it is \$1500
. In the first period, the solar capacity factor is $0.9$ and the wind
capacity factor is $0.45$, but in the second period, there is some
uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are
$0.75$ and $0.5$. Your goal is to identify how to dispatch your
generators to minimize the cost of meeting demand.

#### Problem 2.1

Draw a scenario tree for this problem.

#### Problem 2.2

Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

## References

List any external references consulted, including classmates.